# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor.py](./first_l0_processor.py)
  * YAML file: [first_l0_processor.yaml](./first_l0_processor.yaml)

## Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=3)
# , image="ghcr.io/rs-python/rs-infrastructure-dask-eopf:feat-rspy607-l0-processing" # temp

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

DependencyConflict: requested: "starlette ~= 0.13.0" but found: "starlette 0.45.3"


Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/769d525486594d7fb67117ba8f229225/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/3
Dask workers for 'dask-eopf' are up: 3/3


In [3]:
# Other imports
import getpass
import os
from importlib import reload
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = os.path.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "l0",
)
s3_config = os.path.join(s3_base, "config")
s3_output = os.path.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0_config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
s1_short = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.short.yaml",
    "output_data_dir": f"{s3_output}/s1.short",
}
s1 = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.yaml",
    "output_data_dir": f"{s3_output}/s1",
}
s3 = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_dordop_payload.yaml",
    "output_data_dir": f"{s3_output}/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

17:47:05.039 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/logging_config.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/logging_config.yaml'.

17:47:05.042 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s1/iw_joborder.short.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

17:47:05.043 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s1/iw_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_configuration.yaml'.

17:47:05.044 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s1/iw_joborder.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.yaml'.

17:47:05.045 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s3/l0_processor_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/l0_processor_configuration.yaml'.

17:47:05.046 | INFO    | prefect.S3Bucket - Uploading from 'l0_config/s3/s3_dordop_payload.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'.

17:47:05.074 | INFO    | prefect.S3Bucket - Uploaded 6 files from 'l0_config' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'

In [4]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [10]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/jgaucher/code'


17:50:16.290 | INFO    | prefect.task_runner.dask - Connecting to an existing Dask cluster at tls://127.0.0.1:39151

17:50:16.523 | INFO    | Task run 'hack_payload-449' - Uploaded from '/tmp/tmpc834sdq1' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/.empty'.

17:50:16.532 | INFO    | Task run 'hack_payload-449' - Finished in state Completed()

17:50:18.095 | INFO    | Task run 'all_my_eopf_code-1f8' - INFO:eopf.trigger.local:RUN with {'general_configuration': {'logging': {'level': 'INFO'}, 'triggering__use_basic_logging': True, 'triggering__use_default_filename': False, 'triggering__wait_before_exit': 0, 'dask__export_graphs': './reports/graphs'}, 'workflow': [{'name': 's1_l0_processor', 'active': True, 'module': 'l0.s1.s1_l0_processor', 'processing_unit': 'S1L0Processor', 'inputs': {'CADUS': 'S1ACADU'}, 'outputs': {'datatakeid_.*': 'output_folder'}}], 'I/O': {'input_products': [{'id': 'S1ACADU', 'path': 's3://rs-cluster-temp/stations/CADIP/S1A_20240410083700053369.short', 'store_type': 'cadu', 'store_params': {'storage_options': {'key': '${S3_ACCESSKEY_K8S}', 'secret': '${S3_SECRETKEY_K8S}', 'client_kwargs': {'endpoint_url': '${S3_ENDPOINT_K8S}', 'region_name': '${S3_REGION_K8S}'}}}}], 'output_products': [{'id': 'output_folder', 'path': '${OUTPUT_DIR}/S1A_20240410083700053369.short', 'type': 'folder', 'store_type': 'zarr', 'opening_mode': 'CREATE_OVERWRITE', 'store_params': {'storage_options': {'key': '${S3_ACCESSKEY}', 'secret': '${S3_SECRETKEY}', 'client_kwargs': {'endpoint_url': '${S3_ENDPOINT}', 'region_name': '${S3_REGION}'}}}}]}, 'breakpoints': [], 'dask_context': {'cluster_type': 'gateway', 'cluster_config': {'address': '${DASK_GATEWAY_ADDRESS}', 'reuse_cluster': '${DASK_CLUSTER_NAME}', 'auth': {'type': 'basic', 'username': '${LOCAL_DASK_USERNAME}', 'password': '${LOCAL_DASK_PASSWORD}'}, 'workers': 3}, 'performance_report_file': './reports/report.html'}, 'logging': '../logging_config.yaml', 'config': ['./iw_configuration.yaml']}

...

17:51:04.266 | WARNING | urllib3.connectionpool - Connection pool is full, discarding connection: minio. Connection pool size: 10

17:51:04.273 | WARNING | urllib3.connectionpool - Connection pool is full, discarding connection: minio. Connection pool size: 10

17:51:04.275 | INFO    | Task run 'all_my_eopf_code-1f8' - Uploaded 96 files from 'reports' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/output/s1.short/reports/graphs/S01IWRAW__20240410T075338_0073_A096_T858_678E5_HH.zarr___conditions__echo__IW3__header_flag_header_flag.svg'

17:51:04.279 | INFO    | Task run 'all_my_eopf_code-1f8' - Finished in state Completed()

17:51:04.284 | INFO    | Task run 'single_dpr_task-7c4' - Finished in state Completed()

In [6]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./first_l0_processor.yaml"

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'first-l0-processor/sprint21-first-l0-processor' successfully     │
│ created with id '09a1a764-0757-446a-a320-e7638cc06b7c'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/09a1a764-0757-446a-a320-e7638cc06b7c


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
'first-l0-processor/sprint21-first-l0-processor'



In [7]:
deploy_name = "first-l0-processor/sprint21-first-l0-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'first-l0-processor/sprint21-first-l0-processor'


# TEMP !

In [6]:
# shutdown_dask_clusters(dask_gateway, None)
# init_dask_cluster_eopf(scale=3)
# from resources.utils import *  
# from resources.dask_utils import *  
# from resources.prefect_utils import * 

# # We use only the EOPF dask cluster in this tutorial
# dask_gateway = dask_gateway_eopf
# dask_client = dask_client_eopf
# dask_cluster = dask_cluster_eopf
# if local_mode:
#     os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# # Save cluster info to be read by our flow
# os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

Shutting down cluster '4d3078e211404d408fe5449881bc0975' ...
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/151913d21a6244e0857eb93f759e86a1/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/3
Dask workers for 'dask-eopf' are up: 3/3


In [8]:
# # Import the module, or reload it if you changed its source code
# import first_l0_processor
# reload(first_l0_processor)

# # Run the flow
# results = first_l0_processor.first_l0_processor(**s1_short)
# display(results)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


17:05:32.242 | INFO    | prefect.engine - Created flow run 'amigurumi-emu' for flow 'first-l0-processor'

17:05:32.243 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/d846f8ef-ced5-4c15-8ff2-23f37cbdded8

17:05:32.272 | INFO    | prefect.task_runner.dask - Connecting to existing Dask cluster GatewayCluster<45252f0e8957454c9ecf65e5dfe17712, status=running>

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


...

17:06:14.155 | INFO    | Task run 'single_dpr_task-28e' - Finished in state Completed()

17:06:14.155 | INFO    | Task run 'single_dpr_task-28e' - Finished in state Completed()

17:06:14.155 | INFO    | Task run 'single_dpr_task-28e' - Finished in state Completed()

17:06:14.191 | INFO    | Flow run 'amigurumi-emu' - Finished in state Completed()

None

## Run Prefect flow for S1 short data

In [8]:
output = s1_short["output_data_dir"]
print(f"Remove existing zarr products from: {output!r}")
s3_delete(output)

# Convert to json to trigger prefect flow
s1_short_str = to_json(s1_short)

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'


In [9]:
%%bash -s "$deploy_name" "$s1_short_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor/sprint21-first-l0-processor'...
Created flow run 'peridot-dinosaur'.
└── UUID: 9fec951c-fa25-40fd-b8e3-bab8ff4703cc
└── Parameters: {'input_config_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/config', 'payload_file': 's1/iw_joborder.short.yaml', 'output_data_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'}
└── Job Variables: {}
└── Scheduled start time: 2025-02-25 17:47:39 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/9fec951c-fa25-40fd-b8e3-bab8ff4703cc
Watching flow run 'peridot-dinosaur'...


17:47:39.565 | INFO    | prefect - Flow run is in state 'Scheduled'


17:47:43.423 | INFO    | prefect.task_runner.dask - Connecting to an existing Dask cluster at tls://127.0.0.1:39151

17:47:43.821 | INFO    | Task run 'hack_payload-3d0' - Uploaded from '/tmp/tmpirmili0u' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/output/s1.short/S1A_20240410083700053369.short/.empty'.

17:47:43.831 | INFO    | Task run 'hack_payload-3d0' - Finished in state Completed()

...

17:48:33.755 | INFO    | Task run 'all_my_eopf_code-a5c' - Uploaded 96 files from 'reports' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/output/s1.short/graphs/S01IWRAW__20240410T075338_0073_A096_T524_678E5_HH.zarr___conditions__echo__IW3__sequence_flag_sequence_flag.svg'

17:48:33.762 | INFO    | Task run 'all_my_eopf_code-a5c' - Finished in state Completed()

17:48:33.768 | INFO    | Task run 'single_dpr_task-d94' - Finished in state Completed()

17:48:34.744 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


## Check results

In [ ]:
# Download zarr products into local
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await PREFECT_BLOCK_S3.get_directory(f"{s3_prefix_subdir}", local_dir)
!ls -al "$local_dir"

In [ ]:
# Open them with the zarr python package
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
!pip install zarr
import zarr

for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

In [ ]:
print(f"Output zarr products will be written to: {s3_full_path}")

## 3. Shutdown the dask clusters

In [30]:
if local_mode:

    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

Shutting down cluster '7393a2b3bd404f30a47d693755dcb86a' ...
